In [1]:
# Bird Counter - Jupyter Notebook Version
# Grey-sky optimized: CLAHE contrast boost, dual threshold, lower motion threshold

import cv2
import numpy as np
from collections import defaultdict
import math
import tkinter as tk
from tkinter import filedialog, messagebox
from datetime import datetime
import os
import time
import multiprocessing

# Configuration settings
FRAME_COVERAGE_PERCENTAGE = 0.6
FRAME_SKIP = 1
RESIZE_FACTOR = 0.9
ENABLE_MULTIPROCESSING = False
MAX_WORKERS = max(1, multiprocessing.cpu_count() - 1)

prev_frame = None
motion_threshold = 12   # lowered from 30: grey sky = low contrast = smaller diffs

def profile(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@profile
def detect_moving_birds(current_detections, frame, prev_frame_cache):
    """Motion detection — lower threshold for grey/overcast sky"""
    global prev_frame

    if prev_frame_cache is not None:
        prev_gray = prev_frame_cache
    elif prev_frame is None:
        prev_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return [], prev_frame
    else:
        prev_gray = prev_frame

    current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_diff = cv2.absdiff(prev_gray, current_gray)

    moving_birds = []
    if current_detections:
        for detection in current_detections:
            x, y, w, h = detection['bbox']
            y2 = min(y + h, frame_diff.shape[0])
            x2 = min(x + w, frame_diff.shape[1])
            if y < y2 and x < x2:
                roi_diff = frame_diff[y:y2, x:x2]
                if roi_diff.size > 0:
                    motion_score = np.mean(roi_diff)
                    if motion_score > motion_threshold:
                        moving_birds.append(detection)

    return moving_birds, current_gray


class EnhancedBirdTracker:
    def __init__(self, max_stationary_frames=20, min_movement_distance=15, min_flight_duration=5):
        self.tracks = {}
        self.track_id = 0
        self.max_stationary_frames = max_stationary_frames
        self.min_movement_distance = min_movement_distance
        self.min_flight_duration = min_flight_duration
        self.confirmed_flying_birds = set()
        self.bird_flight_status = {}

    @profile
    def update_tracks(self, detections):
        matched_tracks = {}
        current_frame_birds = []

        active_tracks = {}
        for track_id, track_data in self.tracks.items():
            active_tracks[track_id] = track_data['positions'][-1]

        for detection in detections:
            x, y = detection['center']
            best_match = None
            min_distance = float('inf')

            if len(active_tracks) > 50:
                track_ids = list(active_tracks.keys())
                positions = np.array(list(active_tracks.values()))
                if len(positions) > 0:
                    distances = np.sqrt(((positions[:, 0] - x) ** 2) + ((positions[:, 1] - y) ** 2))
                    min_idx = np.argmin(distances)
                    if distances[min_idx] < 100:
                        min_distance = distances[min_idx]
                        best_match = track_ids[min_idx]
            else:
                for track_id, last_pos in active_tracks.items():
                    if track_id not in matched_tracks:
                        dx = x - last_pos[0]
                        dy = y - last_pos[1]
                        distance = math.sqrt(dx * dx + dy * dy)
                        if distance < min_distance and distance < 100:
                            min_distance = distance
                            best_match = track_id

            if best_match:
                self.tracks[best_match]['positions'].append((x, y))
                self.tracks[best_match]['stationary_count'] = 0
                self.tracks[best_match]['last_seen'] = len(self.tracks[best_match]['positions'])
                matched_tracks[best_match] = detection
                if self.is_bird_flying(best_match):
                    self.confirmed_flying_birds.add(best_match)
                    self.bird_flight_status[best_match] = 'flying'
                    current_frame_birds.append(detection)
            else:
                self.tracks[self.track_id] = {
                    'positions': [(x, y)],
                    'stationary_count': 0,
                    'last_seen': 1,
                    'created_frame': len(list(self.tracks.values())[0]['positions']) if self.tracks else 0
                }
                self.bird_flight_status[self.track_id] = 'new'
                self.track_id += 1

        tracks_to_remove = []
        for track_id in list(self.tracks.keys()):
            if track_id not in matched_tracks:
                self.tracks[track_id]['stationary_count'] += 1
                if self.tracks[track_id]['stationary_count'] >= self.max_stationary_frames:
                    tracks_to_remove.append(track_id)
        for track_id in tracks_to_remove:
            del self.tracks[track_id]
            if track_id in self.bird_flight_status:
                del self.bird_flight_status[track_id]

        return current_frame_birds

    @profile
    def is_bird_flying(self, track_id):
        positions = self.tracks[track_id]['positions']
        if len(positions) < self.min_flight_duration:
            return False

        pos_array = np.array(positions[-10:])
        if len(pos_array) < 2:
            return False

        diffs = np.diff(pos_array, axis=0)
        distances = np.sqrt(np.sum(diffs ** 2, axis=1))
        total_distance = np.sum(distances)
        avg_movement = total_distance / len(distances) if len(distances) > 0 else 0

        movement_consistency = 0
        if len(diffs) >= 2:
            norms = np.sqrt(np.sum(diffs ** 2, axis=1))
            valid_idx = norms > 0
            if np.sum(valid_idx) >= 2:
                norm_diffs = diffs[valid_idx] / norms[valid_idx, np.newaxis]
                dot_products = np.abs(np.dot(norm_diffs, norm_diffs.T))
                mask = ~np.eye(len(norm_diffs), dtype=bool)
                movement_consistency = np.mean(dot_products[mask]) if np.sum(mask) > 0 else 0

        # Relaxed thresholds for grey sky: lower avg_movement and consistency bar
        is_flying = (
            total_distance > self.min_movement_distance and
            avg_movement > 1.0 and       # lowered from 2.0: grey sky = smaller motion signal
            movement_consistency > 0.2   # lowered from 0.3: be more permissive
        )
        return is_flying

    def get_unique_flying_birds_count(self):
        return len(self.confirmed_flying_birds)

    def get_currently_active_birds(self):
        return sum(1 for t in self.tracks if t in self.confirmed_flying_birds)


@profile
def detect_birds_in_frame(frame, frame_height=None, detection_boundary=None):
    """Bird detection with CLAHE + dual threshold for grey/overcast sky"""
    if frame_height is None:
        frame_height = frame.shape[0]
    if detection_boundary is None:
        detection_boundary = int(frame_height * FRAME_COVERAGE_PERCENTAGE)

    roi = frame[:detection_boundary, :]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # CLAHE: boost local contrast — critical for grey sky where birds blend in
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    blurred = cv2.GaussianBlur(gray, (3, 3), 0)

    # Dual threshold: OR of Gaussian + Mean — catches more bird silhouettes on grey sky
    thresh1 = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 11, 3)
    thresh2 = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV, 15, 4)
    thresh = cv2.bitwise_or(thresh1, thresh2)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    detections = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if 8 < area < 1200:   # lowered min (8 from 10), raised max (1200 from 1000)
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / h if h > 0 else 0
            if 0.25 < aspect_ratio < 4.5:  # wider range: grey-sky birds more varied shape
                center_x = x + w // 2
                center_y = y + h // 2
                detection = {
                    'bbox': (x, y, w, h),
                    'center': (center_x, center_y),
                    'area': area,
                    'confidence': min(1.0, area / 100.0)
                }
                detections.append(detection)

    return detections


@profile
def process_video_with_unique_bird_counting(video_path, show_display=False):
    """Optimized video processing — grey-sky tuned"""
    global prev_frame
    prev_frame = None
    prev_frame_cache = None

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return None

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    new_width = int(width * RESIZE_FACTOR)
    new_height = int(height * RESIZE_FACTOR)
    detection_boundary = int(new_height * FRAME_COVERAGE_PERCENTAGE)

    print(f"Processing: {os.path.basename(video_path)}")
    print(f"- Original Resolution: {width}x{height}, Resized: {new_width}x{new_height}")
    print(f"- FPS: {fps}, Frames: {total_frames}, Processing every {FRAME_SKIP} frames")
    print(f"- Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}% of frame")

    bird_tracker = EnhancedBirdTracker(
        max_stationary_frames=int(30 / FRAME_SKIP),
        min_movement_distance=15 * RESIZE_FACTOR,  # lowered from 25: shorter tracks on grey sky
        min_flight_duration=max(2, int(5 / FRAME_SKIP))  # lowered from 8: confirm faster
    )

    back_sub = cv2.createBackgroundSubtractorMOG2(detectShadows=False)

    frame_count = 0
    max_concurrent_birds = 0
    paused = False
    step_mode = False
    processed_frames = 0

    start_time = time.time()
    last_update_time = start_time

    NEON_GREEN = (57, 255, 20)
    WHITE = (255, 255, 255)
    LIGHT_BLUE = (255, 220, 100)

    try:
        while True:
            if not paused or step_mode:
                for _ in range(FRAME_SKIP - 1):
                    ret = cap.grab()
                    if not ret:
                        break

                ret, frame = cap.read()
                if not ret:
                    break

                frame_count += FRAME_SKIP
                processed_frames += 1

                if RESIZE_FACTOR != 1.0:
                    frame = cv2.resize(frame, (new_width, new_height), interpolation=cv2.INTER_AREA)

                roi = frame[:detection_boundary, :]
                fg_mask = back_sub.apply(roi)

                current_detections = detect_birds_in_frame(frame, new_height, detection_boundary)
                moving_detections, prev_frame_cache = detect_moving_birds(current_detections, frame, prev_frame_cache)
                currently_flying_birds = bird_tracker.update_tracks(moving_detections)

                unique_flying_birds = bird_tracker.get_unique_flying_birds_count()
                currently_active = bird_tracker.get_currently_active_birds()
                max_concurrent_birds = max(max_concurrent_birds, currently_active)

                current_time = time.time()
                elapsed = current_time - last_update_time
                if elapsed >= 1.0:
                    fps_estimate = processed_frames / elapsed
                    processed_frames = 0
                    last_update_time = current_time

                if show_display:
                    display_frame = frame.copy()

                    cv2.line(display_frame, (0, detection_boundary), (frame.shape[1], detection_boundary),
                             (255, 255, 0), 2)
                    cv2.putText(display_frame, f"Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}%",
                                (10, detection_boundary + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

                    for detection in currently_flying_birds:
                        x, y, w, h = detection['bbox']
                        cv2.rectangle(display_frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
                        cv2.putText(display_frame, "FLYING", (x, y - 5),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                    progress = frame_count / total_frames
                    cv2.rectangle(display_frame, (5, 5), (480, 145), (0, 0, 0), -1)
                    cv2.rectangle(display_frame, (5, 5), (480, 145), NEON_GREEN, 2)

                    cv2.putText(display_frame, f"Frame: {frame_count}/{total_frames}",
                                (10, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.9, WHITE, 2)
                    cv2.putText(display_frame, f"UNIQUE Flying Birds: {unique_flying_birds}",
                                (10, 78), cv2.FONT_HERSHEY_SIMPLEX, 1.0, NEON_GREEN, 3)
                    cv2.putText(display_frame, f"Currently Active: {currently_active}",
                                (10, 113), cv2.FONT_HERSHEY_SIMPLEX, 0.9, LIGHT_BLUE, 2)
                    cv2.putText(display_frame, f"Max Concurrent: {max_concurrent_birds}",
                                (10, 145), cv2.FONT_HERSHEY_SIMPLEX, 0.9, LIGHT_BLUE, 2)

                    bar_width = 400
                    bar_height = 20
                    bar_x = display_frame.shape[1] - bar_width - 10
                    bar_y = 10
                    cv2.rectangle(display_frame, (bar_x, bar_y),
                                  (bar_x + bar_width, bar_y + bar_height), (100, 100, 100), -1)
                    cv2.rectangle(display_frame, (bar_x, bar_y),
                                  (bar_x + int(bar_width * progress), bar_y + bar_height), NEON_GREEN, -1)
                    cv2.putText(display_frame, f"{progress*100:.1f}%",
                                (bar_x + bar_width // 2 - 20, bar_y + 15),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, WHITE, 2)

                    cv2.imshow("Optimized Bird Counter - Flying Birds Detection", display_frame)

                    key = cv2.waitKey(1 if not paused else 0) & 0xFF
                    if key == ord('q'):
                        break
                    elif key == ord('p'):
                        paused = not paused
                        step_mode = False
                    elif key == ord('s'):
                        step_mode = True
                        paused = False
                    elif key == 27:
                        break

                    if step_mode:
                        paused = True
                        step_mode = False

                if frame_count % (100 * FRAME_SKIP) == 0:
                    progress = frame_count / total_frames
                    elapsed_total = time.time() - start_time
                    remaining = (elapsed_total / progress) - elapsed_total if progress > 0 else 0
                    print(f"  Progress: {frame_count}/{total_frames} frames ({progress*100:.1f}%) - "
                          f"Unique flying birds: {unique_flying_birds} - "
                          f"Est. remaining: {remaining:.1f}s")

        unique_flying_birds = bird_tracker.get_unique_flying_birds_count()
        total_time = time.time() - start_time
        frames_per_second = frame_count / total_time if total_time > 0 else 0

        print(f"  COMPLETED - Unique flying birds: {unique_flying_birds}")
        print(f"  Max concurrent: {max_concurrent_birds}, Total tracks: {bird_tracker.track_id}")
        print(f"  Processing time: {total_time:.2f}s, Speed: {frames_per_second:.2f} frames/s")

        return {
            'video_path': video_path,
            'video_name': os.path.basename(video_path),
            'frames_processed': frame_count,
            'unique_flying_birds': unique_flying_birds,
            'max_concurrent_birds': max_concurrent_birds,
            'total_tracks': bird_tracker.track_id,
            'fps': fps,
            'total_frames': total_frames,
            'duration_seconds': total_frames / fps if fps > 0 else 0,
            'processing_time': total_time,
            'processing_speed': frames_per_second
        }

    except Exception as e:
        print(f"  Error processing video: {str(e)}")
        return None
    finally:
        cap.release()
        if show_display:
            cv2.destroyAllWindows()


def write_results_to_file(results, output_file_path):
    try:
        with open(output_file_path, 'w', encoding='utf-8') as f:
            f.write("BIRD COUNTING RESULTS (OPTIMIZED VERSION)\n")
            f.write("=" * 80 + "\n")
            f.write(f"Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}% of frame\n")
            f.write(f"Frame Skip: {FRAME_SKIP} (processing every {FRAME_SKIP} frames)\n")
            f.write(f"Resize Factor: {RESIZE_FACTOR:.2f}\n")
            f.write(f"Total Videos Processed: {len([r for r in results if r is not None])}\n")
            f.write(f"Processing Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 80 + "\n\n")

            successful_results = [r for r in results if r is not None]
            if successful_results:
                total_birds = sum(r['unique_flying_birds'] for r in successful_results)
                avg_birds = total_birds / len(successful_results)
                max_birds = max(r['unique_flying_birds'] for r in successful_results)
                min_birds = min(r['unique_flying_birds'] for r in successful_results)
                total_processing_time = sum(r['processing_time'] for r in successful_results)
                f.write("SUMMARY STATISTICS:\n")
                f.write("-" * 40 + "\n")
                f.write(f"Total Birds Detected Across All Videos: {total_birds}\n")
                f.write(f"Average Birds per Video: {avg_birds:.1f}\n")
                f.write(f"Maximum Birds in Single Video: {max_birds}\n")
                f.write(f"Minimum Birds in Single Video: {min_birds}\n")
                f.write(f"Total Processing Time: {total_processing_time:.2f} seconds\n\n")

            f.write("INDIVIDUAL VIDEO RESULTS:\n")
            f.write("-" * 40 + "\n")
            for i, result in enumerate(results, 1):
                if result is not None:
                    f.write(f"{i:2d}. {result['video_name']}\n")
                    f.write(f"    Unique Flying Birds: {result['unique_flying_birds']}\n")
                    f.write(f"    Max Concurrent Birds: {result['max_concurrent_birds']}\n")
                    f.write(f"    Total Tracks Created: {result['total_tracks']}\n")
                    f.write(f"    Video Duration: {result['duration_seconds']:.1f} seconds\n")
                    f.write(f"    Frames Processed: {result['frames_processed']}/{result['total_frames']}\n")
                    f.write(f"    Processing Time: {result['processing_time']:.2f} seconds\n")
                    f.write(f"    Processing Speed: {result['processing_speed']:.2f} frames/second\n")
                    f.write(f"    File Path: {result['video_path']}\n\n")
                else:
                    f.write(f"{i:2d}. [FAILED TO PROCESS]\n\n")

            f.write("\nCSV FORMAT (for spreadsheet import):\n")
            f.write("-" * 40 + "\n")
            f.write("Video_Name,Unique_Flying_Birds,Max_Concurrent,Total_Tracks,Duration_Seconds,Processing_Time,Speed_FPS\n")
            for result in results:
                if result is not None:
                    f.write(f"{result['video_name']},{result['unique_flying_birds']},"
                            f"{result['max_concurrent_birds']},{result['total_tracks']},"
                            f"{result['duration_seconds']:.1f},{result['processing_time']:.2f},"
                            f"{result['processing_speed']:.2f}\n")

        print(f"\nResults written to: {output_file_path}")
        return True
    except Exception as e:
        print(f"Error writing results to file: {str(e)}")
        return False


def select_video_folder():
    root = tk.Tk()
    root.withdraw()
    folder_path = filedialog.askdirectory(title="Select Folder Containing Video Files")
    root.destroy()
    return folder_path

def get_video_files(folder_path):
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv', '.webm', '.m4v', '.mpg', '.mpeg']
    video_files = []
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        if os.path.isfile(file_path):
            _, ext = os.path.splitext(file_path)
            if ext.lower() in video_extensions:
                video_files.append(file_path)
    return video_files

def process_single_video():
    root = tk.Tk()
    root.withdraw()
    filetypes = [("Video files", "*.mp4 *.avi *.mov *.mkv *.wmv *.flv *.webm"), ("All files", "*.*")]
    video_path = filedialog.askopenfilename(title="Select Video File for Bird Counting", filetypes=filetypes)
    root.destroy()
    if not video_path:
        print("No video file selected. Exiting...")
        return
    print(f"Selected video: {os.path.basename(video_path)}")
    try:
        result = process_video_with_unique_bird_counting(video_path, show_display=True)
        if result:
            print(f"\n{'='*60}\nFINAL RESULTS\n{'='*60}")
            print(f"Video: {result['video_name']}")
            print(f"TOTAL UNIQUE FLYING BIRDS: {result['unique_flying_birds']}")
            print(f"Maximum concurrent flying birds: {result['max_concurrent_birds']}")
            print(f"Processing time: {result['processing_time']:.2f} seconds")
            print(f"Processing speed: {result['processing_speed']:.2f} frames/second")
            print(f"{'='*60}")
    except Exception as e:
        print(f"Error processing video: {str(e)}")
        messagebox.showerror("Error", f"Failed to process video:\n{str(e)}")

def process_batch_videos():
    folder_path = select_video_folder()
    if not folder_path:
        print("No folder selected. Exiting...")
        return
    video_files = get_video_files(folder_path)
    if not video_files:
        print(f"No video files found in folder: {folder_path}")
        return
    print(f"\nFound {len(video_files)} video files in: {folder_path}")
    for i, vf in enumerate(video_files, 1):
        print(f"  {i:2d}. {os.path.basename(vf)}")
    try:
        confirm = input(f"\nProcess all {len(video_files)} videos? (y/n): ").strip().lower()
        if confirm not in ['y', 'yes']:
            print("Processing cancelled.")
            return
    except KeyboardInterrupt:
        print("\nProcessing cancelled.")
        return

    results = []
    total_birds_all_videos = 0
    successful_count = 0
    failed_count = 0
    print(f"\n{'='*60}\nSTARTING BATCH PROCESSING\n{'='*60}")
    start_time = datetime.now()

    for i, video_path in enumerate(video_files, 1):
        print(f"\n[{i}/{len(video_files)}] Processing: {os.path.basename(video_path)}")
        print("-" * 50)
        try:
            result = process_video_with_unique_bird_counting(video_path, show_display=False)
            if result is not None:
                results.append(result)
                total_birds_all_videos += result['unique_flying_birds']
                successful_count += 1
                print(f"  SUCCESS - {result['unique_flying_birds']} birds detected in {result['processing_time']:.2f}s")
            else:
                results.append(None)
                failed_count += 1
        except Exception as e:
            print(f"  ERROR - {str(e)}")
            results.append(None)
            failed_count += 1

    processing_time = datetime.now() - start_time
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(folder_path, f"bird_count_results_{os.path.basename(folder_path)}_{timestamp}.txt")
    write_results_to_file(results, output_path)
    print(f"\n{'='*60}\nBATCH PROCESSING COMPLETED\n{'='*60}")
    print(f"Total Videos: {len(video_files)}")
    print(f"Successfully Processed: {successful_count}")
    print(f"Failed: {failed_count}")
    print(f"Total Processing Time: {processing_time}")
    print(f"TOTAL BIRDS DETECTED ACROSS ALL VIDEOS: {total_birds_all_videos}")
    if successful_count > 0:
        print(f"Average Birds per Video: {total_birds_all_videos/successful_count:.1f}")
    print(f"{'='*60}")

def run_bird_counter():
    print("Optimized Bird Counter - Jupyter Notebook Version")
    print("=" * 60)
    print(f"Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}% of frame")
    print(f"Frame Skip: {FRAME_SKIP}")
    print(f"Resize Factor: {RESIZE_FACTOR:.2f}")
    print("Grey-sky mode: CLAHE + dual threshold + lower motion sensitivity")
    print()
    print("Processing Options:")
    print("1. Batch process all videos in a folder (no display)")
    print("2. Process single video with display")
    while True:
        try:
            choice = input("Enter your choice (1 or 2): ").strip()
            if choice in ['1', '2']:
                break
            else:
                print("Please enter 1 or 2")
        except KeyboardInterrupt:
            print("\nExiting...")
            return
    if choice == '2':
        process_single_video()
    else:
        process_batch_videos()

#Takes video properties -> calculates new dimensions for resizing -> adjust the minimum flight duration, the number of stationary frames -> subtract the background -> start timer -> detect_birds_inFrame() ----> calculate boundaries for only the region of interest -> convert to gray -> CLAHE contrast boost -> gaussian blur -> dual adaptive thresholding (Gaussian OR Mean) -> morphological operations -> finding contours -> filter contours and draw bounding rectangles -> detect_moving_birds(observed_birds) --- -> convert frame to gray -> calculate frame difference -> check if there is significant motion (lower threshold for grey sky) -> update moving birds in the video -> get the count of the total unique birds.

In [2]:
# NOTEBOOK-FRIENDLY VERSION - No tkinter dialogs
# Video path is set below:

VIDEO_PATH = "C_7142025_2 (online-video-cutter.com) (1).mp4"

In [3]:
# Process a single video (set show_display=False for notebooks)
import os

if os.path.exists(VIDEO_PATH):
    print(f"Processing: {VIDEO_PATH}")
    result = process_video_with_unique_bird_counting(VIDEO_PATH, show_display=False)
    
    if result:
        print(f"\n{'='*60}")
        print(f"FINAL RESULTS")
        print(f"{'='*60}")
        print(f"Video: {result['video_name']}")
        print(f"TOTAL UNIQUE FLYING BIRDS: {result['unique_flying_birds']}")
        print(f"Maximum concurrent flying birds: {result['max_concurrent_birds']}")
        print(f"Processing time: {result['processing_time']:.2f} seconds")
        print(f"Processing speed: {result['processing_speed']:.2f} frames/second")
        print(f"{'='*60}")
else:
    print(f"ERROR: Video file not found at: {VIDEO_PATH}")
    print("Please update VIDEO_PATH in the cell above to point to your video file.")

ERROR: Video file not found at: C_7142025_2 (online-video-cutter.com) (1).mp4
Please update VIDEO_PATH in the cell above to point to your video file.
